In [91]:
import pandas as pd

In [92]:
df = pd.read_csv('laptops.csv')

In [93]:
df.head()

,name,processor,ram,storage,display,graphics,prices
0,Lenovo Legion Pro 5 AI Powered Gaming Laptop,Intel Core Ultra 7,32GB RAM,1TB SSD,16 Inch (40.64 cm) WQXGA OLED Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"2,14,999"
1,Dell Alienware 16 Aurora Gaming Laptop,Intel Core 7,16 GB RAM,1TB SSD,16 Inch (40.64 cm) WQXGA Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"1,59,990"
2,Acer Aspire Lite AL15-41 Thin and Light Laptop,AMD Ryzen 3,8 GB RAM,512GB SSD,15.6 Inch (39.62 cm) FHD Display,AMD Radeon Graphics,"52,599"
3,Samsung Galaxy Book 6,Intel Core Ultra 7,16GB RAM,512GB SSD,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,31,990"
4,Samsung Galaxy Book 6,Intel Core Ultra 5,16GB RAM,512GB SSD,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,23,990"


## Observations

### Model Name

### Processor 

### Ram

### Storage

### Display

### Graphics

* format is wrong in nvideas graphics card
* not availble for rows - 21, 149, 150, 151, 152, 153, 228, 229, 250, 254, 255, 281, 282, 283, 284, 285, 303, 304, 305, 306, 307, 308, 309, 310, 333, 334, 335, 336, 337, 338, 339, 340,  352, 356, 380, 
* nvidia format right for rows - 74, 81, 88, 93, 96, 131, 222, 370, 377, 378, 384, 
* integrated cards in rows - 220, 221, 223, 
* data shifte by 1 for rows - 106,  168, 175, 200, 201, 235, 373, 387, 

### Splitting the Brand name from the name column

In [94]:
df.insert(loc=1, column='Brand', value=df['name'].str.split().str[0])

### shifting the columns by 1 for macbooks

In [95]:
cols = ['processor', 'ram', 'storage', 'display', 'graphics']

df.loc[df['name'].str.contains('MacBook'), cols] = (
    df.loc[df['name'].str.contains('MacBook'), cols].shift(1, axis=1)
)

### getting processor for macs

In [96]:
df.loc[df['name'].str.contains('MacBook'), 'processor'] = (
    df.loc[df['name'].str.contains('MacBook'), 'name']
    .str.extract(r'((?:M[1-5]|A\d+)(?:\s*(?:Pro|Max|Ultra))?)', expand=False)
)

### Droping the Refurbished and store display units

In [ ]:
store_list = df[df['name'].str.contains('Store|Refurbished')].index
df = df.drop(index=store_list)

### correcting intel processor format problem 

In [98]:
processor_format_problem = df[df['processor'].str.startswith('1')].index
snap_list = df[df['processor'].str.contains('Snapdragon')].index
intel_problem = list(set(processor_format_problem) - set(snap_list))

In [99]:
df[df['processor'].str.contains('13650HX|1334U|13420H|1305U|13450HX|1355U|1305U|12650H')]['processor'].index

Index([143, 228, 231, 240, 279, 293, 295, 297, 314, 320, 323, 325, 327, 344,
       350],
      dtype='int64')

In [100]:
df.loc[[143, 228, 231, 240, 279, 293, 295, 297, 314, 320, 323, 325, 327, 344,
       350], 'processor'] = df.loc[[143, 228, 231, 240, 279, 293, 295, 297, 314, 320, 323, 325, 327, 344,
       350], 'processor'].str.split('-').str[0]

In [101]:
df.loc[intel_problem]['processor'].value_counts()

processor
13th Gen Intel Core i5    28
13th Gen Intel Core i7    19
13th Gen Intel Core i3     8
14th Gen Intel Core i7     5
13th Gen Core i5           2
13th Gen Core i3           2
12th Gen Intel Core i5     2
12th Gen Core i5           1
14th Gen Intel Core i5     1
12th Gen Intel Core i7     1
Name: count, dtype: int64

In [102]:
df[df['processor'].str.contains('13th Gen Core i5|3th Gen Core i3|12th Gen Core i5')]['processor'].index

Index([154, 161, 320, 350, 383], dtype='int64')

In [103]:
temp = df.loc[[154, 161, 320, 350, 383], 'processor'].str.split()
temp

154    [13th, Gen, Core, i5]
161    [12th, Gen, Core, i5]
320    [13th, Gen, Core, i3]
350    [13th, Gen, Core, i3]
383    [13th, Gen, Core, i5]
Name: processor, dtype: object

In [104]:
df.loc[[154, 161, 320, 350, 383], 'processor'] = (
    temp.apply(lambda x: ' '.join(x[:2] + ['Intel'] + x[2:]))
)

In [105]:
temp = df.loc[intel_problem, 'processor'].str.split()
temp

256    [14th, Gen, Intel, Core, i7]
257    [13th, Gen, Intel, Core, i7]
258    [13th, Gen, Intel, Core, i3]
8      [13th, Gen, Intel, Core, i7]
9      [13th, Gen, Intel, Core, i5]
                   ...             
122    [13th, Gen, Intel, Core, i5]
127    [13th, Gen, Intel, Core, i5]
125    [13th, Gen, Intel, Core, i7]
254    [13th, Gen, Intel, Core, i7]
383    [13th, Gen, Intel, Core, i5]
Name: processor, Length: 69, dtype: object

In [106]:
df.loc[intel_problem, 'processor'] = (
    temp.apply(lambda x: ' '.join(x[2:] + x[:2]))
)

In [107]:
df['processor'].value_counts()

processor
Intel Core Ultra 7              40
Intel Core i5 13th Gen          36
M5                              21
Intel Core Ultra 5              20
AMD Ryzen 7                     20
                                ..
Snapdragon® X X1-26-100          1
Intel Core Ultra 5 Series 2      1
Intel Core Ultra 5 225H          1
Intel Core Ultra 9 285H          1
Intel Core Ultra 9               1
Name: count, Length: 69, dtype: int64

### dropping rows with snapdragon processor

In [108]:
df = df.drop(index=snap_list)

In [109]:
df['processor'].value_counts()

processor
Intel Core Ultra 7              40
Intel Core i5 13th Gen          36
M5                              21
Intel Core Ultra 5              20
AMD Ryzen 7                     20
Intel Core i7 13th Gen          19
Intel Core 5                    14
AMD Ryzen 5                     12
Intel Core i3 13th Gen          11
AMD Ryzen AI 5                   9
AMD Ryzen 3                      8
Intel Core 3                     8
A18 Pro                          7
Intel Core i5                    6
M4 Pro                           6
Intel Core 7                     6
Intel Core i7 14th Gen           6
M4                               5
M2                               5
AMD Ryzen AI 7                   5
M3 Chip                          5
Intel Core Ultra 9               4
Intel Core i5 12th Gen           3
AMD Ryzen 5 Quad Core 7520U      3
Intel Core Ultra X7              3
M4 Max                           2
M3                               2
AMD Ryzen 7-7435HS               2
Intel Core

In [110]:
def parse_processor(proc):
    if pd.isna(proc):
        return pd.Series([None, None, None])
    
    proc = proc.replace('™', '').strip()
    
    # Apple
    if proc.startswith(('M1','M2','M3','M4','M5','A1')):
        return pd.Series(['Apple', proc.replace(' Chip','').strip(), None])
    
    # Intel
    if proc.startswith('Intel'):
        if 'Ultra' in proc:
            series = 'Core Ultra'
        elif 'Core i' in proc:
            series = 'Core i'
        else:
            series = 'Core'
        
        # tier: Ultra 7, Core i5, Core 5 etc.
        tier = None
        match = re.search(r'(?:Ultra\s*)?([i\d]?\d)', proc)
        if match:
            tier = match.group(1)
        
        return pd.Series(['Intel', series, tier])
    
    # AMD
    if proc.startswith('AMD'):
        if 'Ryzen AI' in proc:
            series = 'Ryzen AI'
        else:
            series = 'Ryzen'
        
        tier = None
        match = re.search(r'Ryzen(?:\s*AI)?\s*(\d)', proc)
        if match:
            tier = match.group(1)
        
        return pd.Series(['AMD', series, tier])
    
    return pd.Series([None, None, None])

import re
df[['processor_brand', 'processor_series', 'processor_tier']] = df['processor'].apply(parse_processor)

In [111]:
processor_brand = df.pop('processor_brand')
processor_series = df.pop('processor_series')
processor_tier = df.pop('processor_tier')

df.insert(3, 'processor_brand', processor_brand)
df.insert(4, 'processor_series', processor_series)
df.insert(5, 'processor_tier', processor_tier)

In [112]:
df.head()

,name,Brand,processor,processor_brand,processor_series,processor_tier,ram,storage,display,graphics,prices
0,Lenovo Legion Pro 5 AI Powered Gaming Laptop,Lenovo,Intel Core Ultra 7,Intel,Core Ultra,7,32GB RAM,1TB SSD,16 Inch (40.64 cm) WQXGA OLED Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"2,14,999"
1,Dell Alienware 16 Aurora Gaming Laptop,Dell,Intel Core 7,Intel,Core,7,16 GB RAM,1TB SSD,16 Inch (40.64 cm) WQXGA Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"1,59,990"
2,Acer Aspire Lite AL15-41 Thin and Light Laptop,Acer,AMD Ryzen 3,AMD,Ryzen,3,8 GB RAM,512GB SSD,15.6 Inch (39.62 cm) FHD Display,AMD Radeon Graphics,"52,599"
3,Samsung Galaxy Book 6,Samsung,Intel Core Ultra 7,Intel,Core Ultra,7,16GB RAM,512GB SSD,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,31,990"
4,Samsung Galaxy Book 6,Samsung,Intel Core Ultra 5,Intel,Core Ultra,5,16GB RAM,512GB SSD,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,23,990"


### Correcting Ram format

In [113]:
df['ram'].value_counts()

ram
 16GB RAM              104
16GB RAM                24
 16GB DDR5 RAM          22
 16GB DDR4 RAM          21
 32GB RAM               20
 8GB RAM                16
 24GB RAM               15
 16GB LPDDR5X RAM       14
 16GB LPDDR5x RAM       14
 16 GB RAM              13
24GB RAM                12
8GB RAM                  8
 8 GB RAM                6
 32GB LPDDR5X RAM        5
 8GB LPDDR5 RAM          4
 16GB LPDDR5 RAM         4
 16GB RAM                3
 8GB RAM                 3
 8GB DDR4 RAM            3
 8GB DDR5 RAM            2
 12GB RAM                2
 24GB DDR5 RAM           2
48GB RAM                 2
 16GB DDR5-4800 RAM      2
 32GB DDR5 RAM           1
36GB RAM                 1
 16GB                    1
 32GB LPDDR5x RAM        1
 16 GB LPDDR4X RAM       1
 8GB                     1
2022) (8GB RAM           1
Name: count, dtype: int64

In [114]:
df['ram'] = df['ram'].str.extract(r'(\d+)')

In [115]:
df = df.drop(index=[413])

### correcting storage column format

In [116]:
df['storage'].value_counts()

storage
 512GB SSD        203
 1TB SSD           89
 512GB Storage      9
 256GB SSD          8
 1TB Storage        6
 512 GB SSD         5
 2TB SSD            2
512GB SSD           2
 512 GB SSD         1
512GB SSD           1
512 GB SSD          1
Name: count, dtype: int64

In [117]:
df['storage'] = df['storage'].str.extract(r'(\d+\s*(?:GB|TB))')

In [118]:
df['storage'] = df['storage'].str.extract(r'(\d+)\s*(GB|TB)').apply(lambda x: x[0] + x[1], axis=1)

In [119]:
df.head()

,name,Brand,processor,processor_brand,processor_series,processor_tier,ram,storage,display,graphics,prices
0,Lenovo Legion Pro 5 AI Powered Gaming Laptop,Lenovo,Intel Core Ultra 7,Intel,Core Ultra,7,32,1TB,16 Inch (40.64 cm) WQXGA OLED Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"2,14,999"
1,Dell Alienware 16 Aurora Gaming Laptop,Dell,Intel Core 7,Intel,Core,7,16,1TB,16 Inch (40.64 cm) WQXGA Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"1,59,990"
2,Acer Aspire Lite AL15-41 Thin and Light Laptop,Acer,AMD Ryzen 3,AMD,Ryzen,3,8,512GB,15.6 Inch (39.62 cm) FHD Display,AMD Radeon Graphics,"52,599"
3,Samsung Galaxy Book 6,Samsung,Intel Core Ultra 7,Intel,Core Ultra,7,16,512GB,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,31,990"
4,Samsung Galaxy Book 6,Samsung,Intel Core Ultra 5,Intel,Core Ultra,5,16,512GB,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,23,990"


### Dropping the prev splitted columns

In [120]:
df = df.drop(columns=['name', 'processor'])

In [121]:
df.head()

,Brand,processor_brand,processor_series,processor_tier,ram,storage,display,graphics,prices
0,Lenovo,Intel,Core Ultra,7,32,1TB,16 Inch (40.64 cm) WQXGA OLED Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"2,14,999"
1,Dell,Intel,Core,7,16,1TB,16 Inch (40.64 cm) WQXGA Display,8GB-NVIDIA GeForce RTX 5060 Graphics,"1,59,990"
2,Acer,AMD,Ryzen,3,8,512GB,15.6 Inch (39.62 cm) FHD Display,AMD Radeon Graphics,"52,599"
3,Samsung,Intel,Core Ultra,7,16,512GB,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,31,990"
4,Samsung,Intel,Core Ultra,5,16,512GB,16 Inch (40.62 cm) WUXGA Display with Touchsc...,Intel Graphics,"1,23,990"


In [122]:
df['display'].value_counts()

display
 15.6 Inch (39.62 cm) FHD Display                  68
 15.6 Inch (39.62 cm) Full HD Display              21
 13 Inch Liquid Retina Display                     19
 16 Inch (40.64 cm) FHD+ Display                    9
 14.2 inch (35.97 cm) Liquid Retina XDR Display     8
                                                   ..
14 Inch (35.56) FHD Display                         1
 15.6 Inch (39.62 cm) Full HD IPS Display           1
 14 Inch (35.6 cm) FHD+ Display                     1
 14 Inch (35.56cm) WUXGA Display                    1
 14 inch (35.56 cm) Liquid Retina XDR Display       1
Name: count, Length: 93, dtype: int64

In [123]:
df.loc[[106, 235, 387]]

,Brand,processor_brand,processor_series,processor_tier,ram,storage,display,graphics,prices
106,Dell,AMD,Ryzen AI,5,16,512GB,16 Inch (40.64 cm) 2K Anti-Glare Display,AMD Radeon™ Graphics,"72,950"
235,Lenovo,AMD,Ryzen,5,16,512GB,15.6 Inch (39.62 cm) FHD Display,AMD Radeon Graphics,"42,990"
387,Apple,Apple,M4 Pro,None,24,512GB,16.2 inch (41.05 cm) Liquid Retina XDR Display,14-core CPU,"2,29,900"


In [124]:
df['screen_size'] = df['display'].str.split().str[0]

In [125]:
df['screen_size'].value_counts()

screen_size
15.6         132
14            71
16            56
13            19
15.3          10
15             8
14.2           8
13.3           5
16.2           5
8              5
13.6           3
Full           1
AMD            1
Windows        1
NVIDIA         1
13.6-inch      1
Name: count, dtype: int64

In [126]:
df.reset_index(drop=True, inplace=True)

In [127]:
df[df['display'].str.contains('Windows|NVIDIA|13.6-inch|Core|AMD')].index

Index([101, 212, 285, 306, 307, 308, 309, 310, 311], dtype='int64')

In [139]:
df[df['display'].str.contains('Windows|NVIDIA|13.6-inch|Core|AMD')]

,Brand,processor_brand,processor_series,processor_tier,ram,storage,display,graphics,prices,screen_size
101,HP,AMD,Ryzen,5,8,512GB,Windows 11 Home,AMD Radeon Graphics,"49,999",Windows


In [129]:
df = df.drop(index=311)

In [131]:
temp = df.loc[[101, 212, 285, 306, 307, 308, 309, 310], 'display']
df.loc[[101, 212, 285, 306, 307, 308, 309, 310], 'display'] = df.loc[[101, 212, 285, 306, 307, 308, 309, 310], 'graphics']
df.loc[[101, 212, 285, 306, 307, 308, 309, 310], 'graphics'] = temp

In [134]:
df = df.drop(index=[212])

In [140]:
df = df.drop(index=[101])

In [145]:
df = df.drop(index=88)

In [160]:
# screen size
df['screen_size'] = df['display'].str.extract(r'(\d+\.?\d*)\s*[Ii]nch').astype(float)

# display type
df['display_type'] = df['display'].str.extract(r'(FHD\+?|Full HD|WUXGA|WQXGA\+?|OLED|AMOLED|IPS|2\.5[Kk]|2K|3K|4K|QHD|Liquid Retina XDR|Liquid Retina|PixelSense|AMOLED 2X)')

In [163]:
screen_size = df.pop('screen_size')
display_type = df.pop('display_type')

df.insert(6, 'screen_size', screen_size)
df.insert(7, 'display_type', display_type)

In [167]:
df['display_type'].value_counts()

display_type
FHD                  99
Full HD              42
Liquid Retina        37
WUXGA                29
FHD+                 20
WQXGA+               16
Liquid Retina XDR    14
OLED                 13
WQXGA                12
2K                    8
3K                    6
2.5K                  5
2.5k                  4
Name: count, dtype: int64

In [173]:
df.loc[(df['display_type'].isna()) & (df['screen_size'] == 15.6), 'display_type'] = 'FHD'

In [177]:
df = df.drop(index=[105, 143, 153, 154, 199, 218, 254, 268, 271])

In [184]:
df = df.drop(columns=['display'])

In [185]:
df.head()

,Brand,processor_brand,processor_series,processor_tier,ram,storage,screen_size,display_type,graphics,prices
0,Lenovo,Intel,Core Ultra,7,32,1TB,16.0,WQXGA,8GB-NVIDIA GeForce RTX 5060 Graphics,"2,14,999"
1,Dell,Intel,Core,7,16,1TB,16.0,WQXGA,8GB-NVIDIA GeForce RTX 5060 Graphics,"1,59,990"
2,Acer,AMD,Ryzen,3,8,512GB,15.6,FHD,AMD Radeon Graphics,"52,599"
3,Samsung,Intel,Core Ultra,7,16,512GB,16.0,WUXGA,Intel Graphics,"1,31,990"
4,Samsung,Intel,Core Ultra,5,16,512GB,16.0,WUXGA,Intel Graphics,"1,23,990"


In [186]:
df['graphics'].value_counts()

graphics
Intel Graphics                          40
Intel UHD Graphics                      31
AMD Radeon Graphics                     24
Intel Arc Graphics                      18
10‑core CPU and 10‑core GPU             16
                                        ..
6GB RTX™ 3050 Graphics                   1
AMD XDNA NPU up to 50TOPS                1
8GB-NVIDIA GeForce RTX 5070 Graphics     1
Integrated AMD Radeon Graphics GPU       1
Intel Arc 140T GPU                       1
Name: count, Length: 76, dtype: int64

In [201]:
df[df['graphics'].str.contains('NVIDIA')]['graphics'].value_counts()

graphics
6GB-NVIDIA GeForce RTX 4050 Graphics           12
6GB-NVIDIA GeForce RTX 3050 Graphics           12
8GB-NVIDIA GeForce RTX 5050 Graphics           10
8GB-NVIDIA GeForce RTX 5060 Graphics            9
4GB-NVIDIA GeForce RTX 3050 Graphics            4
6GB NVIDIA GeForce RTX 3050 Graphics            3
NVIDIA RTX 5060 8GB Graphics                    3
NVIDIA GeForce RTX 3050 4GB Graphics            2
NVIDIA GeForce RTX 5060 8GB GPU                 2
16GB-NVIDIA GeForce RTX 5080 Graphics           2
4GB-NVIDIA GeForce RTX 3050A Graphics           2
6 GB-NVIDIA GeForce RTX TM 3050 Graphics        2
8GB NVIDIA GeForce RTX 5060 Graphics            2
NVIDIA GeForce RTX 3050 6GB Graphics            1
NVIDIA GeForce RTX 5050 8GB Graphics            1
6GB NVIDIA GeForce RTX 4050 Graphics            1
NVIDIA GeForce RTX 5060 8GB Graphics            1
NVIDIA GeForce RTX2050 4GB Graphics             1
4GB NVIDIA GeForce RTX 3050A Graphics           1
4GB GDDR6 NVIDIA GeForce RTX 3050A Graphi

In [206]:
df[df['graphics'].str.strip().str.match(r'^\d')]['graphics'].value_counts()

graphics
10‑core CPU and 10‑core GPU                    16
6GB-NVIDIA GeForce RTX 4050 Graphics           12
6GB-NVIDIA GeForce RTX 3050 Graphics           12
8GB-NVIDIA GeForce RTX 5050 Graphics           10
8GB-NVIDIA GeForce RTX 5060 Graphics            9
6‑core CPU and 5‑core GPU                       7
10-core CPU                                     6
8 Core CPU 10 CORE GPU                          5
14-core CPU                                     5
8-core CPU                                      5
10‑core CPU and 8‑core GPU                      4
4GB-NVIDIA GeForce RTX 3050 Graphics            4
6GB NVIDIA GeForce RTX 3050 Graphics            3
12-core CPU                                     2
6GB RTX 4050 Graphics                           2
6 GB-NVIDIA GeForce RTX TM 3050 Graphics        2
4GB-NVIDIA GeForce RTX 3050A Graphics           2
8GB NVIDIA GeForce RTX 5060 Graphics            2
16GB-NVIDIA GeForce RTX 5080 Graphics           2
4GB NVIDIA GeForce RTX 3050A Graphics    